# 00: Environment check and raw data contract

It reads Parquet **footers only**. No row group is decompressed, so the whole notebook completes in a couple of seconds against 86.6 M rows.

## Setup

Find the repository root by walking upwards, then put it on `sys.path` so both packages import cleanly regardless of where
Jupyter was started.

In [1]:
# Reload edited .py modules without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "oem_analysis").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from oem_analysis.config import se_config as C
from oem_analysis.lib import se_store
from oem_analysis.lib import se_diagnostics as diag

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Repository root:", REPO_ROOT)
print("Data root:      ", C.DATA_ROOT)

Repository root: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA
Data root:       C:\Users\z3553082\OneDrive - UNSW\Documents\CICCADA - Data\solar edge


## 1. Environment

Required packages must be present. 

Optional ones are informational.

Paths must resolve. 

In [2]:
env = diag.environment_report()
display(env)

missing = env.loc[env["required"] & ~env["pass"]]
if len(missing):
    raise RuntimeError(f"Required items missing:\n{missing[['item', 'detail']]}")
print("Environment OK.")

,item,status,detail,required,pass
0,package: duckdb,ok,1.5.4,True,True
1,package: pandas,ok,3.0.3,True,True
2,package: numpy,ok,2.4.6,True,True
3,package: pyarrow,ok,24.0.0,False,True
4,package: matplotlib,ok,3.11.0,False,True
5,package: seaborn,ok,0.13.2,False,True
6,package: geopandas,ok,1.1.4,False,True
7,package: shapely,ok,2.1.2,False,True
8,package: pytz,ok,2026.2,False,True
9,path: REPO_ROOT,ok,C:\Users\z3553082\OneDrive - UNSW\Documents\Gi...,True,True


Environment OK.


## 2. Conventions

Everything the ingest step will apply to turn telemetry into the common CICCADA convention. 

This dataset seems to use the conventions:

- **Active power** is reported as a production magnitude: Over all of 2025 it has `min = 0` and no negative values, so it is already generator-positive. No change.
- **Reactive power** ASSUMED reported in the **generator convention**, where  *negative = absorbing*. AS/NZS 4777.2 Fig 3.2 use the **generatorconvention**, where *negative = absorbing*. So `Q` is multiplied by `+1` at ingest.


[**BMS Note after running notebook 03**: Some sites showcase adverse response with correct magnitude. Is this a case of inverter logging the sign on a different convention? Or genuine adversity?]

In [3]:
display(C.describe_conventions())

,convention,value
0,active power: source convention,"generator (production magnitude, always >= 0)"
1,active power: sign applied,+1 (no change)
2,reactive sign: sites fitting,"213 as-delivered / 106 flipped / 1,271 neither"
3,reactive power: source convention,AS DELIVERED -- majority of curve-following si...
4,reactive power: sign applied,+1 (no change)
5,target convention,generator (AS/NZS 4777.2 Fig 3.2; negative Q =...
6,basis for the reactive sign,"fleet_orientation_fit over all 1,590 assessabl..."
7,active power units,W -> kW
8,reactive power units,var -> kvar (instantaneous; NOT multiplied by 12)
9,raw timestamp frame,"per-site local civil time, INCLUDING daylight ..."


## 3. Connect

An in-memory DuckDB connection. 

No data is copied into a database file.

The store stays as Parquet on disk, readable by pandas, polars and Arrow, and regenerable from the raw delivery.

`se_raw` spans all 12 monthly files as a single relation. `se_alias` is the site mapping CSV. Store tables are registered only once they have been built.

In [ ]:
con = se_store.connect(verbose=True)
display(se_store.relations(con))

  registered  se_raw               12 monthly Parquet files
  registered  se_alias             alias_mapping_alias_only.csv
  registered  se_interval          partitioned, 24 files, + derived columns


,relation,table_type
0,se_alias,VIEW
1,se_interval,VIEW
2,se_raw,VIEW


## 4. Raw inventory

Footer metadata for each delivered file.

Two columns matter beyond the row counts:

- **`schema_fingerprint`**: A hash of the ordered `(column, type)` list. One distinct value across all 12 files means one schema, so a single query can read the lot.
- **`n_row_groups`**: every file has exactly **one** row group holding millions of rows. There are therefore no row-group statistics to prune on, and any predicate forces a full column scan.

In [ ]:
inv = diag.raw_inventory(con)
display(inv[["month", "n_rows", "n_row_groups", "rows_per_row_group",
             "n_columns", "size_mb", "schema_fingerprint"]])

print(f"Files:        {len(inv)}")
print(f"Total rows:   {inv.n_rows.sum():,}")
print(f"Total size:   {inv.size_mb.sum():,.0f} MB compressed on disk")
print(f"Writer:       {inv.created_by.iloc[0]}")
print()
print("For scale: as float64 in pandas this would be roughly "
      f"{inv.n_rows.sum() * 14 * 8 / 1024**3:.1f} GB in memory, "
      "which is why nothing is ever loaded whole.")

,month,n_rows,n_row_groups,rows_per_row_group,n_columns,size_mb,schema_fingerprint
0,2025-01,8232278,1,8232278,15,144.7,4da382b05e99
1,2025-02,7090717,1,7090717,15,125.9,4da382b05e99
2,2025-03,7218954,1,7218954,15,127.6,4da382b05e99
3,2025-04,6619100,1,6619100,15,117.6,4da382b05e99
4,2025-05,6439017,1,6439017,15,114.0,4da382b05e99
5,2025-06,6074226,1,6074226,15,107.9,4da382b05e99
6,2025-07,6422544,1,6422544,15,113.6,4da382b05e99
7,2025-08,6783051,1,6783051,15,120.1,4da382b05e99
8,2025-09,7059218,1,7059218,15,125.2,4da382b05e99
9,2025-10,7864708,1,7864708,15,139.0,4da382b05e99


Files:        12
Total rows:   86,643,185
Total size:   1,533 MB compressed on disk
Writer:       parquet-cpp-arrow version 6.0.1

For scale: as float64 in pandas this would be roughly 9.0 GB in memory, which is why nothing is ever loaded whole.


## 5. Raw schema

The 15 delivered columns. `se_config.RAW_COLUMNS` is the declared contract.

**absent**: no nameplate capacity, no inverter model, no DNSP, no install date, and no irradiance. 

Site metadata is postcode and state only.

In [ ]:
schema = diag.raw_schema(con, Path(inv.path.iloc[0]))
schema["declared"] = schema.column_name.map(C.RAW_COLUMNS)
display(schema)

,column_name,column_type,declared
0,site_alias,VARCHAR,string
1,timestamp,VARCHAR,string
2,active_power_1,FLOAT,float
3,active_power_2,FLOAT,float
4,active_power_3,FLOAT,float
5,reactive_power_1,FLOAT,float
6,reactive_power_2,FLOAT,float
7,reactive_power_3,FLOAT,float
8,ac_voltage_1,FLOAT,float
9,ac_voltage_2,FLOAT,float


## 6. D1 checks

Three groups of assertions:

1. **Schema contract**: all files share one schema, and it is the declared one.
2. **Inventory**: 12 months with no gaps, and per-file row counts matching `se_config.EXPECTED_RAW_ROWS` (measured 12 Aug 2026).
3. **Alias mapping**: complete, unique, and every state has a timezone mapped.

In [ ]:
checks = diag.run_d1_checks(con, inventory=inv)
display(checks)

ok = diag.summarise(checks, label="D1")
assert ok, "D1 checks failed. see above before proceeding to D2."

,group,check,expected,observed,pass,note
0,schema contract,all files share one schema,1 distinct fingerprint,1 (4da382b05e99),True,
1,schema contract,column names and order,15 columns,15 columns,True,
2,schema contract,column types match contract,0 mismatches,0 mismatches,True,
3,inventory,months present,"12 months, 2025-01..2025-12",12 months,True,
4,inventory,per-file row counts,12 exact matches,12 matches,True,
5,inventory,total rows,"86,643,185","86,643,185",True,
6,inventory,row-group structure,informational,12/12 files have a single row group,True,Single row groups mean no statistics to prune ...
7,alias mapping,alias mapping rows,"1,602","1,602",True,
8,alias mapping,aliases are unique,"1,602 distinct","1,602 distinct",True,
9,alias mapping,postcodes present,0 nulls,0 nulls,True,


D1: PASS  (11/11 checks)


## 7. Site mapping

The complete site metadata: 1,602 aliases, each with a postcode and a state.

The 1,602 sites resolve to a few hundred distinct postcodes, so the BOM irradiance extract in (which is keyed on the nearest BOM
grid point, not on the site) will be far smaller than the site count suggests.

In [ ]:
display(diag.alias_mapping_summary(con))

postcodes = se_store.q(con, "SELECT count(DISTINCT zip_code) AS n_postcodes FROM se_alias")
print(f"Distinct postcodes across the fleet: {int(postcodes.n_postcodes.iloc[0])}")
print("These collapse further into BOM grid points at D4, which sets the D12a extract size.")

,state,n_sites,n_postcodes,first_alias,last_alias
0,South Australia,574,153,AUS002,AUS998
1,New South Wales,570,202,AUS001,AUS999
2,Queensland,458,152,AUS007,AUS994


Distinct postcodes across the fleet: 507
These collapse further into BOM grid points at D4, which sets the D12a extract size.


## 8. Store status

What has been built so far. Everything is expected to be missing at this point.

This table is the SolarEdge analogue of `conformance_queries.table_provenance()`: it
makes it impossible to run an analysis against a store you believed was complete but
was not.

In [ ]:
display(se_store.store_status(con)[["logical_name", "exists", "kind", "size_mb", "n_rows"]])

,logical_name,exists,kind,size_mb,n_rows
0,se_interval,True,partitioned,1470.6,86640968.0
1,se_interval_phase,False,partitioned,NaN,NaN
2,se_site,False,file,NaN,NaN
3,se_site_capacity,False,file,NaN,NaN
4,bom_solar,False,file,NaN,NaN
5,se_structured,False,partitioned,NaN,NaN
6,se_ghi_model,False,file,NaN,NaN
7,se_uncurtailedpv,False,partitioned,NaN,NaN
